# Dataset Downloader

This notebook downloads datasets based on a CSV file input containing dataset information and video keys.

## Features:
- Configurable storage folder
- Test mode (downloads only 3 rows)
- Progress tracking
- Error handling

In [1]:
# Import required libraries
import pandas as pd
import os
from pathlib import Path
from tqdm import tqdm
import time
from lerobot.datasets.lerobot_dataset import LeRobotDataset

## Configuration

Set up the configuration parameters:
- `CSV_FILE_PATH`: Path to your CSV file containing dataset information
- `STORAGE_FOLDER`: Where to store downloaded datasets
- `VIDEO_KEY_COLUMN`: Column name containing the video key
- `TEST_MODE`: Set to True to download only 3 rows for testing

In [8]:
# Configuration
BASE_PATH ="/home/michelmeyer/Dev/LeRobotLab"
CSV_FILE_PATH = BASE_PATH + "/csv/test-datasets-cristian.csv"  # Update this path
STORAGE_FOLDER = BASE_PATH + "/downloaded_datasets"
VIDEO_KEY_COLUMN = "video_key"  # Update this column name
DATASET_COLUMN = "dataset"  # Column containing the dataset name (username/foldername)
TEST_MODE = True  # Set to False for full download

# Create storage folder if it doesn't exist
Path(STORAGE_FOLDER).mkdir(parents=True, exist_ok=True)
print(f"Storage folder: {STORAGE_FOLDER}")
print(f"CSV File Path: {Path(CSV_FILE_PATH).resolve()}")
print(f"Test mode: {'ON (3 rows only)' if TEST_MODE else 'OFF'}")

Storage folder: /home/michelmeyer/Dev/LeRobotLab/downloaded_datasets
CSV File Path: /home/michelmeyer/Dev/LeRobotLab/csv/test-datasets-cristian.csv
Test mode: ON (3 rows only)


## Load and Preview Dataset

In [9]:
# Load the CSV file
try:
    df = pd.read_csv(CSV_FILE_PATH)
    print(f"Loaded CSV with {len(df)} rows")
    print(f"Columns: {list(df.columns)}")
    print("\nFirst 5 rows:")
    print(df.head())
    
    # Verify required columns exist
    required_columns = [VIDEO_KEY_COLUMN, DATASET_COLUMN]
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"Missing columns: {missing_columns}. Available columns: {list(df.columns)}")
        
except FileNotFoundError:
    print(f"Error: CSV file not found at {CSV_FILE_PATH}")
    print("Please update the CSV_FILE_PATH variable with the correct path")
except Exception as e:
    print(f"Error loading CSV: {e}")

Loaded CSV with 4 rows
Columns: ['dataset', 'video_key']

First 5 rows:
                       dataset        video_key
0        1lyz123576/so101_test            phone
1       smanni/train_so100_all  intel_realsense
2  bjb7/so101_pen_touch_test_1         camera_4
3   shreyasgite/so100_base_env           laptop


## Download Function

Function to download a single dataset using the Hugging Face LeRobot API.

In [10]:
def download_dataset(video_key, dataset, storage_folder, row_index=None):
    """
    Download dataset using LeRobot
    
    Args:
        video_key: The video key identifier
        dataset: Dataset name in format "username/foldername"
        storage_folder: Base folder to store downloads
        row_index: Optional row index for progress tracking
    
    Returns:
        dict: Download result with status and details
    """
    try:
        # Parse username and foldername from dataset
        username, foldername = dataset.split('/')
        
        # Create folder structure: storage_folder/username/foldername
        dataset_folder = Path(storage_folder) / username / foldername
        dataset_folder.mkdir(parents=True, exist_ok=True)
        
        print(f"  Downloading dataset: {dataset}")
        print(f"  Video key: {video_key}")
        print(f"  Target folder: {dataset_folder}")
        
        # Load/download dataset using LeRobot
        dataset_obj = LeRobotDataset(dataset, root=str(dataset_folder))
                
        return {
            'status': 'success',
            'video_key': video_key,
            'dataset': dataset,
            'folder': str(dataset_folder),
            'dataset_length': len(dataset_obj),
            'message': f"Downloaded {dataset} ({len(dataset_obj)} episodes) to {dataset_folder}"
        }
        
    except Exception as e:
        print(f"  Error downloading dataset {dataset}: {e}")
        return {
            'status': 'error',
            'video_key': video_key,
            'dataset': dataset,
            'folder': None,
            'dataset_length': 0,
            'message': f"Error downloading {dataset}: {str(e)}"
        }


## Main Download Process

Process the CSV file and download datasets for each video key.

In [12]:
# Prepare data for download
if 'df' in locals():
    # Apply test mode if enabled
    if TEST_MODE:
        download_df = df.head(3).copy()
        print(f"Test mode: Processing only {len(download_df)} rows")
    else:
        download_df = df.copy()
        print(f"Full mode: Processing all {len(download_df)} rows")
    
    # Initialize results tracking
    results = []
    successful_downloads = 0
    failed_downloads = 0
    
    print(f"\nStarting download process...")
    print(f"Datasets to process:")
    for _, row in download_df.iterrows():
        print(f"  - Video key: {row[VIDEO_KEY_COLUMN]}, Dataset: {row[DATASET_COLUMN]}")
    
    # Process each row
    for index, row in tqdm(download_df.iterrows(), total=len(download_df), desc="Downloading datasets"):
        video_key = row[VIDEO_KEY_COLUMN]
        dataset = row[DATASET_COLUMN]
        
        print(f"\nProcessing row {index + 1}:")
        
        # Download dataset
        result = download_dataset(video_key, dataset, STORAGE_FOLDER, index)
        results.append(result)
        
        # Track success/failure
        if result['status'] == 'success':
            successful_downloads += 1
            print(f"✓ Success: {result['message']}")
        else:
            failed_downloads += 1
            print(f"✗ Failed: {result['message']}")
        
        # Small delay between downloads
        time.sleep(1)
    
    # Summary
    print(f"\n{'='*50}")
    print(f"DOWNLOAD SUMMARY")
    print(f"{'='*50}")
    print(f"Total processed: {len(download_df)}")
    print(f"Successful: {successful_downloads}")
    print(f"Failed: {failed_downloads}")
    print(f"Storage location: {STORAGE_FOLDER}")
    
else:
    print("No data loaded. Please fix the CSV loading issue first.")

Test mode: Processing only 3 rows

Starting download process...
Datasets to process:
  - Video key: phone, Dataset: 1lyz123576/so101_test
  - Video key: intel_realsense, Dataset: smanni/train_so100_all
  - Video key: camera_4, Dataset: bjb7/so101_pen_touch_test_1



Processing row 1:
  Video key: phone
  Target folder: /home/michelmeyer/Dev/LeRobotLab/downloaded_datasets/1lyz123576/so101_test


Resolving data files:   0%|          | 0/135 [00:00<?, ?it/s]

✓ Success: Downloaded 1lyz123576/so101_test (64505 episodes) to /home/michelmeyer/Dev/LeRobotLab/downloaded_datasets/1lyz123576/so101_test



Processing row 2:
  Video key: intel_realsense
  Target folder: /home/michelmeyer/Dev/LeRobotLab/downloaded_datasets/smanni/train_so100_all


...

✓ Success: Downloaded smanni/train_so100_all (510641 episodes) to /home/michelmeyer/Dev/LeRobotLab/downloaded_datasets/smanni/train_so100_all



Processing row 3:
  Video key: camera_4
  Target folder: /home/michelmeyer/Dev/LeRobotLab/downloaded_datasets/bjb7/so101_pen_touch_test_1


Resolving data files:   0%|          | 0/100 [00:00<?, ?it/s]

✓ Success: Downloaded bjb7/so101_pen_touch_test_1 (21648 episodes) to /home/michelmeyer/Dev/LeRobotLab/downloaded_datasets/bjb7/so101_pen_touch_test_1



DOWNLOAD SUMMARY
Total processed: 3
Successful: 3
Failed: 0
Storage location: /home/michelmeyer/Dev/LeRobotLab/downloaded_datasets


## Results Analysis

Analyze the download results and show detailed information.

In [7]:
# Create results DataFrame for analysis
if 'results' in locals() and results:
    results_df = pd.DataFrame(results)
    
    print("Download Results:")
    print(results_df[['status', 'video_key', 'dataset', 'dataset_length', 'message']])
    
    # Show successful downloads
    successful_results = results_df[results_df['status'] == 'success']
    if not successful_results.empty:
        print(f"\nSuccessful downloads ({len(successful_results)}):")
        for _, result in successful_results.iterrows():
            print(f"  - {result['video_key']} ({result['dataset']}): {result['dataset_length']} episodes")
    
    # Show failed downloads
    failed_results = results_df[results_df['status'] == 'error']
    if not failed_results.empty:
        print(f"\nFailed downloads ({len(failed_results)}):")
        for _, result in failed_results.iterrows():
            print(f"  - {result['video_key']} ({result['dataset']}): {result['message']}")
    
    # List downloaded folders with username/foldername structure
    print(f"\nContents of storage folder {STORAGE_FOLDER}:")
    storage_path = Path(STORAGE_FOLDER)
    if storage_path.exists():
        for username_dir in storage_path.iterdir():
            if username_dir.is_dir():
                print(f"  📁 {username_dir.name}/")
                for dataset_dir in username_dir.iterdir():
                    if dataset_dir.is_dir():
                        print(f"    📁 {dataset_dir.name}/")
    else:
        print("  Storage folder not found")

else:
    print("No results to analyze. Run the download process first.")

Download Results:
    status video_key                dataset  dataset_length  \
0  success     phone  1lyz123576/so101_test           64505   

                                             message  
0  Downloaded 1lyz123576/so101_test (64505 episod...  

Successful downloads (1):
  - phone (1lyz123576/so101_test): 64505 episodes

Contents of storage folder /home/michelmeyer/Dev/LeRobotLab/downloaded_datasets:
  📁 lerobot/
    📁 aloha_sim_insertion_human/
  📁 1lyz123576/
    📁 so101_test/
  📁 smanni/
    📁 train_so100_all/
  📁 newtechmitch/
    📁 orange-ball-3/
    📁 orange-ball-1/


## Next Steps

To use this notebook with LeRobot datasets:

1. **Install Requirements**: 
   ```bash
   pip install lerobot huggingface_hub torch
   ```

2. **Prepare CSV File**: 
   Your CSV should have columns:
   - `video_key`: Identifier for the specific video/episode
   - `repo_id`: HuggingFace repository ID (e.g., "lerobot/pusht", "lerobot/aloha_sim_insertion_human")

3. **Update Configuration**: 
   - Set the correct `CSV_FILE_PATH`
   - Update column names if different
   - Set `STORAGE_FOLDER` to your preferred location

4. **Authentication** (if needed):
   ```bash
   huggingface-cli login
   ```

5. **Test First**: 
   - Keep `TEST_MODE = True` for initial testing
   - Only set to `False` when ready for full download

**Example CSV format:**
```csv
video_key,repo_id
episode_001,lerobot/pusht
episode_002,lerobot/aloha_sim_insertion_human
episode_003,lerobot/xarm_lift_medium
```

## Download Status Check

Check if all datasets from the CSV file have been successfully downloaded to the storage folder.

In [ ]:
def check_downloaded_datasets(csv_file_path, storage_folder, dataset_column, video_key_column):
    """
    Check which datasets from the CSV have been downloaded
    
    Args:
        csv_file_path: Path to the CSV file
        storage_folder: Base storage folder
        dataset_column: Column name containing dataset names
        video_key_column: Column name containing video keys
    
    Returns:
        dict: Summary of download status
    """
    try:
        # Load CSV
        df = pd.read_csv(csv_file_path)
        storage_path = Path(storage_folder)
        
        downloaded = []
        missing = []
        
        print(f"Checking download status for {len(df)} datasets...")
        print(f"Storage folder: {storage_path}")
        
        for _, row in df.iterrows():
            dataset = row[dataset_column]
            video_key = row[video_key_column]
            
            # Parse username/foldername
            username, foldername = dataset.split('/')
            expected_path = storage_path / username / foldername
            
            if expected_path.exists() and expected_path.is_dir():
                # Check if folder has content
                content = list(expected_path.glob("*"))
                if content:
                    downloaded.append({
                        'video_key': video_key,
                        'dataset': dataset,
                        'path': str(expected_path),
                        'files_count': len(content)
                    })
                else:
                    missing.append({
                        'video_key': video_key,
                        'dataset': dataset,
                        'path': str(expected_path),
                        'reason': 'Empty folder'
                    })
            else:
                missing.append({
                    'video_key': video_key,
                    'dataset': dataset,
                    'path': str(expected_path),
                    'reason': 'Folder not found'
                })
        
        return {
            'total': len(df),
            'downloaded': downloaded,
            'missing': missing,
            'download_rate': len(downloaded) / len(df) * 100 if len(df) > 0 else 0
        }
        
    except Exception as e:
        return {'error': str(e)}

# Run the check
print("=" * 60)
print("DOWNLOAD STATUS CHECK")
print("=" * 60)

status = check_downloaded_datasets(CSV_FILE_PATH, STORAGE_FOLDER, DATASET_COLUMN, VIDEO_KEY_COLUMN)

if 'error' in status:
    print(f"Error checking downloads: {status['error']}")
else:
    print(f"Total datasets in CSV: {status['total']}")
    print(f"Successfully downloaded: {len(status['downloaded'])}")
    print(f"Missing/Failed: {len(status['missing'])}")
    print(f"Download completion rate: {status['download_rate']:.1f}%")
    
    if status['downloaded']:
        print(f"\nDownloaded datasets ({len(status['downloaded'])}):")
        for item in status['downloaded']:
            print(f"  - {item['video_key']} ({item['dataset']}): {item['files_count']} files")
    
    if status['missing']:
        print(f"\nMissing datasets ({len(status['missing'])}):")
        for item in status['missing']:
            print(f"  - {item['video_key']} ({item['dataset']}): {item['reason']}")
        
        print(f"\nTo download missing datasets, update your CSV to include only missing ones or re-run the download process.")

## Dataset Integrity Check

Check the integrity of downloaded datasets by attempting to load them and verify their structure.

In [ ]:
def check_dataset_integrity(dataset_path, dataset_name):
    """
    Check if a downloaded dataset can be properly loaded and has valid structure
    
    Args:
        dataset_path: Path to the dataset folder
        dataset_name: Name of the dataset (username/foldername)
    
    Returns:
        dict: Integrity check results
    """
    try:
        # Try to load the dataset
        dataset = LeRobotDataset(dataset_name, root=str(dataset_path))
        
        # Basic integrity checks
        dataset_length = len(dataset)
        
        # Check if we can access the first episode
        if dataset_length > 0:
            first_episode = dataset[0]
            features = list(first_episode.keys()) if hasattr(first_episode, 'keys') else []
        else:
            features = []
        
        # Check for common required files/folders
        dataset_folder = Path(dataset_path)
        has_videos = any(dataset_folder.rglob("*.mp4"))
        has_data_files = any(dataset_folder.rglob("*.json")) or any(dataset_folder.rglob("*.parquet"))
        
        return {
            'status': 'valid',
            'dataset_length': dataset_length,
            'features': features,
            'has_videos': has_videos,
            'has_data_files': has_data_files,
            'message': f"Dataset loaded successfully with {dataset_length} episodes"
        }
        
    except Exception as e:
        return {
            'status': 'invalid',
            'dataset_length': 0,
            'features': [],
            'has_videos': False,
            'has_data_files': False,
            'message': f"Failed to load dataset: {str(e)}"
        }

def check_all_datasets_integrity(csv_file_path, storage_folder, dataset_column, video_key_column):
    """
    Check integrity of all downloaded datasets
    """
    try:
        df = pd.read_csv(csv_file_path)
        storage_path = Path(storage_folder)
        
        integrity_results = []
        
        print(f"Checking integrity of downloaded datasets...")
        
        for _, row in tqdm(df.iterrows(), total=len(df), desc="Checking integrity"):
            dataset = row[dataset_column]
            video_key = row[video_key_column]
            
            # Parse username/foldername
            username, foldername = dataset.split('/')
            dataset_path = storage_path / username / foldername
            
            if dataset_path.exists():
                print(f"\nChecking {dataset}...")
                integrity = check_dataset_integrity(dataset_path, dataset)
                integrity['video_key'] = video_key
                integrity['dataset'] = dataset
                integrity['path'] = str(dataset_path)
                integrity_results.append(integrity)
            else:
                integrity_results.append({
                    'status': 'missing',
                    'video_key': video_key,
                    'dataset': dataset,
                    'path': str(dataset_path),
                    'dataset_length': 0,
                    'features': [],
                    'has_videos': False,
                    'has_data_files': False,
                    'message': 'Dataset folder not found'
                })
        
        return integrity_results
        
    except Exception as e:
        print(f"Error during integrity check: {e}")
        return []

# Run integrity check
print("=" * 60)
print("DATASET INTEGRITY CHECK")
print("=" * 60)

integrity_results = check_all_datasets_integrity(CSV_FILE_PATH, STORAGE_FOLDER, DATASET_COLUMN, VIDEO_KEY_COLUMN)

if integrity_results:
    valid_datasets = [r for r in integrity_results if r['status'] == 'valid']
    invalid_datasets = [r for r in integrity_results if r['status'] == 'invalid']
    missing_datasets = [r for r in integrity_results if r['status'] == 'missing']
    
    print(f"Total datasets checked: {len(integrity_results)}")
    print(f"Valid datasets: {len(valid_datasets)}")
    print(f"Invalid datasets: {len(invalid_datasets)}")
    print(f"Missing datasets: {len(missing_datasets)}")
    
    if valid_datasets:
        print(f"\nValid datasets ({len(valid_datasets)}):")
        for result in valid_datasets:
            print(f"  - {result['video_key']} ({result['dataset']}): {result['dataset_length']} episodes")
    
    if invalid_datasets:
        print(f"\nInvalid datasets ({len(invalid_datasets)}):")
        for result in invalid_datasets:
            print(f"  - {result['video_key']} ({result['dataset']}): {result['message']}")
    
    if missing_datasets:
        print(f"\nMissing datasets ({len(missing_datasets)}):")
        for result in missing_datasets:
            print(f"  - {result['video_key']} ({result['dataset']}): {result['message']}")
            
    # Create summary DataFrame
    summary_df = pd.DataFrame(integrity_results)
    print(f"\nIntegrity Summary:")
    print(summary_df[['video_key', 'dataset', 'status', 'dataset_length', 'has_videos', 'has_data_files']])
    
else:
    print("No datasets found to check integrity.")